# CIFAR-10: attention-head fault tolerance in a ViT-style encoder

Replication of the `transformers_EXP_A` protocol on a **real** image corpus.

The token embedding is replaced by a 4x4 patch projection (8x8 = 64 patch tokens plus CLS); the fault model, criticality definition and safe-region safe-region analysis are unchanged.

**Dataset**: CIFAR-10, 50 000 training / 10 000 test images, 10 classes (`cifar-10-python.tar.gz`, md5 `c58f30108f718f92721af3b95e74349a`). This is the same corpus the Dropout-and-Outliers paper uses for its ViT, so the two studies become directly comparable.

**Model**: fixed *Base* configuration - 4 encoder layers x 8 heads = **32 attention heads**.

A crashed head here removes a channel of *spatial patch* attention rather than syntactic or temporal attention, which makes this the strongest test of whether the layer-position pattern is universal.

**Research questions** (as in the source study)

- **RQ1** Do Transformer models contain critical and redundant attention heads?
- **RQ2** Is degradation explained by the number of failed heads alone, or also by which heads fail?
- **RQ3** How do the fault-tolerance results compare across datasets and modalities, given one fixed architecture?

**Hypotheses**

- **H1** Attention-head criticality is non-uniform across the architecture.
- **H2** Layer-wise fragility is non-monotonic and task-dependent.
- **H3** Degradation increases with the number of failed attention heads.
- **H4** Criticality-aware fault severity adds information beyond the raw fault count.
- **H5** Configurations containing more critical heads are likelier to exceed an acceptable degradation threshold.

## Imports

In [1]:
import json, pathlib, sys, time

import numpy as np
import pandas as pd
import torch

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from faulttol import config as C
from faulttol import multihead, plots, saferegion, single_head
from faulttol.injection import collect_attention_stats_per_head, get_all_heads
from faulttol.runtime import configure_threads, device_report, select_device
from faulttol.training import make_loader

print("torch", torch.__version__, "| threads", configure_threads(),
      "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("root:", ROOT)

torch 2.14.0+cu126 | threads 8 | cuda build 12.6 | cuda available True
root: D:\Laboratory\fault-tolerance-models-and-task-comparison


## Global variables

In [2]:
RESULTS_DIR = ROOT / "results" / "base_transformer" / "cifar10"
MODEL_NAME = "base_transformer_cifar10"
ARCHIVE = ROOT / "data" / "cifar-10-python.tar.gz"
# NVIDIA GPU when one is visible; FT_DEVICE=cpu reproduces the CPU-only run
DEVICE = select_device()

IMAGE_SIZE, PATCH_SIZE, NUM_CLASSES = 32, 4, 10
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
STD = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32)

MODEL_CONFIG = dict(C.BASE_PARAM, activation="relu")

TRAIN_CONFIG = {
    "loss": "CrossEntropyLoss",
    "optimizer": "AdamW",
    # a from-scratch ViT on 32x32 images needs a larger step and stronger decay
    # than the EXP_A synthetic task; cosine annealing follows the ViT recipe of
    # the Dropout-and-Outliers paper, which uses this same corpus
    "learning_rate": 6e-4,
    "weight_decay": 5e-2,
    "batch_size": 128,
    "epochs": C._env_int("FT_EPOCHS", 40),
    "patience": C._env_int("FT_PATIENCE", 8),
    "grad_clip": 1.0,
    "label_smoothing": 0.1,
    "selection_metric": "roc_auc",
    "cosine_schedule": True,
    "augmentation": "RandomCrop(32, padding=4) + RandomHorizontalFlip",
}

CRIT_COL = "D_pos_mean"
DEG_COL = "D_mean"
SAFE_COL = "D_pos_mean"
DEG_LABEL = "mean probability damage"
SAFE_LABEL = "mean positive damage"
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

print("device:", device_report(DEVICE))
print("patches %dx%d = %d tokens | epochs %d"
      % (IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE,
         (IMAGE_SIZE // PATCH_SIZE) ** 2, TRAIN_CONFIG["epochs"]))

device: cuda:0 NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB | sm_89 | torch cuda 12.6 | tf32 off
patches 8x8 = 64 tokens | epochs 40


In [3]:
OUT = pathlib.Path(RESULTS_DIR)
TABLES, FIGURES, MODELS = OUT / "tables", OUT / "figures", OUT / "models"
for d in (TABLES, FIGURES, MODELS):
    d.mkdir(parents=True, exist_ok=True)

def save(df, name):
    path = TABLES / name
    df.to_csv(path, index=False)
    print("saved", path.relative_to(ROOT), df.shape)
    return path

print("artifacts ->", OUT)
print("K_TRIALS=%d  N_FAULT=%d" % (C.K_TRIALS, C.N_FAULT))

artifacts -> D:\Laboratory\fault-tolerance-models-and-task-comparison\results\base_transformer\cifar10
K_TRIALS=10000  N_FAULT=1000


## STEP 1: CIFAR-10 dataset

In [4]:
import pickle, tarfile

def load_cifar10():
    train_batches, test_batch = [], None
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        for member in tar.getmembers():
            name = pathlib.PurePosixPath(member.name).name
            if name.startswith("data_batch_"):
                train_batches.append((name, pickle.load(tar.extractfile(member), encoding="bytes")))
            elif name == "test_batch":
                test_batch = pickle.load(tar.extractfile(member), encoding="bytes")
    train_batches.sort(key=lambda kv: kv[0])

    def to_arrays(batch):
        x = batch[b"data"].reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
        y = np.asarray(batch[b"labels"], dtype=np.int64)
        return (x - MEAN[None, :, None, None]) / STD[None, :, None, None], y

    xs, ys = zip(*[to_arrays(b) for _, b in train_batches])
    tx, ty = to_arrays(test_batch)
    return np.concatenate(xs), np.concatenate(ys), tx, ty

train_x, train_y, test_x, test_y = load_cifar10()
print("raw: train=%s test=%s" % (train_x.shape, test_x.shape))
print("class balance:", np.bincount(train_y))

raw: train=(50000, 3, 32, 32) test=(10000, 3, 32, 32)
class balance: [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]


In [5]:
import torch.nn.functional as F

def augment_batch(xb):
    """RandomCrop(32, padding=4) + RandomHorizontalFlip, batched.

    Vectorised via advanced indexing: with the model on the GPU a per-sample
    Python loop here is the slowest part of the epoch.  The random draws are
    taken in the same order and shape as the loop version, so the augmentation
    stream is unchanged.
    """
    b = xb.size(0)
    padded = F.pad(xb, (4, 4, 4, 4), mode="reflect")
    ox = torch.randint(0, 9, (b,))
    oy = torch.randint(0, 9, (b,))
    flip = torch.rand(b) < 0.5

    ar = torch.arange(32)
    rows = (oy.view(b, 1) + ar.view(1, 32)).view(b, 1, 32, 1)
    # a horizontal flip is the crop read with its columns reversed
    cols = torch.where(flip.view(b, 1), (31 - ar).view(1, 32), ar.view(1, 32))
    cols = (ox.view(b, 1) + cols).view(b, 1, 1, 32)

    return padded[torch.arange(b).view(b, 1, 1, 1),
                  torch.arange(3).view(1, 3, 1, 1),
                  rows, cols]

def balanced_subset(y, n_total, seed):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    per = n_total // len(classes)
    picks = [rng.choice(np.flatnonzero(y == c), size=per, replace=False) for c in classes]
    out = np.concatenate(picks)
    rng.shuffle(out)
    return out

rng = np.random.default_rng(C.SPLIT_SEED)
perm = rng.permutation(len(train_x))
n_val = 5_000
val_idx, tr_idx = perm[:n_val], perm[n_val:]

n_sub = C._env_int("FT_TRAIN_SUBSAMPLE", 0)
if n_sub:
    tr_idx = tr_idx[:n_sub]

X_train, y_train = torch.from_numpy(train_x[tr_idx]), train_y[tr_idx]
X_val, y_val = torch.from_numpy(train_x[val_idx]), train_y[val_idx]

X_test_all = torch.from_numpy(test_x)
eval_idx = balanced_subset(test_y, C.N_FAULT, C.FAULT_SPLIT_SEED)
calib_mask = np.ones(len(test_y), dtype=bool)
calib_mask[eval_idx] = False
X_calib, y_calib = X_test_all[calib_mask], test_y[calib_mask]
X_eval, y_eval = X_test_all[eval_idx].contiguous(), test_y[eval_idx]

print("train=%d val=%d calib=%d fault_eval=%d"
      % (len(X_train), len(X_val), len(X_calib), len(X_eval)))
print("fault-eval class balance:", np.bincount(y_eval))

train=45000 val=5000 calib=9000 fault_eval=1000
fault-eval class balance: [100 100 100 100 100 100 100 100 100 100]


## STEP 2: encoder-only Vision Transformer

In [6]:
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.model import ImageClassifier, count_params
from faulttol.training import train_classifier

torch.manual_seed(C.TRAIN_SEED)
np.random.seed(C.TRAIN_SEED)

model = ImageClassifier(MODEL_CONFIG, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                        in_channels=3, num_classes=NUM_CLASSES, activation="relu")
print("params = %d" % count_params(model))
print("tokens = %d patches (+CLS)" % model.n_patches)
print("heads  = %d" % len(get_all_heads(model)))
model

params = 3174410
tokens = 64 patches (+CLS)
heads  = 32


ImageClassifier(
  (pos): PositionalEncoding()
  (layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (W_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.05, inplace=False)
        (3): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
  )
  (norm_out): Identity()
  (dropout): Dropout(p=0.05, inplace=False)
  (patch): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
  (class

In [7]:
hist = train_classifier(
    model, X_train, y_train, X_val, y_val, DEVICE,
    lr=TRAIN_CONFIG["learning_rate"], weight_decay=TRAIN_CONFIG["weight_decay"],
    epochs=TRAIN_CONFIG["epochs"], batch_size=TRAIN_CONFIG["batch_size"],
    patience=TRAIN_CONFIG["patience"], grad_clip=TRAIN_CONFIG["grad_clip"],
    ckpt_path=MODELS / "best.pt",
    config={"model": MODEL_CONFIG, "train": TRAIN_CONFIG},
    selection_metric=TRAIN_CONFIG["selection_metric"],
    cosine=TRAIN_CONFIG["cosine_schedule"],
    label_smoothing=TRAIN_CONFIG["label_smoothing"],
    augment=augment_batch,
)
save(pd.DataFrame(hist["history"]), "training_history.csv")
pd.DataFrame(hist["history"])

    epoch  1  loss 1.8236  val_acc 0.4902  val_f1 0.4947  val_auc 0.8875  (15s)


    epoch  2  loss 1.5485  val_acc 0.5620  val_f1 0.5633  val_auc 0.9166  (14s)


    epoch  3  loss 1.4577  val_acc 0.5936  val_f1 0.5858  val_auc 0.9280  (14s)


    epoch  4  loss 1.3989  val_acc 0.6064  val_f1 0.6037  val_auc 0.9353  (14s)


    epoch  5  loss 1.3520  val_acc 0.6476  val_f1 0.6435  val_auc 0.9414  (14s)


    epoch  6  loss 1.3116  val_acc 0.6524  val_f1 0.6494  val_auc 0.9431  (14s)


... (30 outputs left out)

    epoch 37  loss 0.7233  val_acc 0.8158  val_f1 0.8139  val_auc 0.9779  (15s)


    early stopping at epoch 37


saved results\base_transformer\cifar10\tables\training_history.csv (37, 8)


,epoch,train_loss,secs,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,1,1.823605,14.796363,0.4902,0.525154,0.489389,0.494682,0.887513
1,2,1.548479,14.412705,0.5620,0.579082,0.562319,0.563325,0.916617
2,3,1.457720,14.439403,0.5936,0.596924,0.593895,0.585809,0.927992
3,4,1.398886,14.441720,0.6064,0.630884,0.607802,0.603733,0.935272
4,5,1.351971,14.432243,0.6476,0.652326,0.647200,0.643477,0.941413
5,6,1.311632,14.464578,0.6524,0.656842,0.651064,0.649426,0.943125
6,7,1.284151,14.471611,0.6630,0.672627,0.664337,0.658876,0.947677
7,8,1.252537,14.452198,0.6818,0.685731,0.682372,0.677594,0.953825
8,9,1.217344,14.456250,0.6744,0.700456,0.673849,0.677057,0.954088
9,10,1.193805,14.470795,0.7032,0.704030,0.702746,0.700090,0.957964


In [8]:
calib_metrics = classification_clean_metrics(
    classification_outputs(model, X_calib, y_calib, DEVICE))

def evaluate():
    return classification_outputs(model, X_eval, y_eval, DEVICE, chunk=500)

damage_fn = classification_damage
clean_eval = classification_clean_metrics(evaluate())

t0 = time.time()
for _ in range(5):
    evaluate()
per_eval = (time.time() - t0) / 5.0

print("calibration set (n=%d):" % len(X_calib), calib_metrics)
print("fixed eval subset (n=%d):" % len(X_eval), clean_eval)
print("cost: %.3f s per configuration -> %.1f min for K=%d"
      % (per_eval, per_eval * C.K_TRIALS / 60.0, C.K_TRIALS))

attn_stats = collect_attention_stats_per_head(
    model, make_loader(X_eval, torch.as_tensor(y_eval), 100, shuffle=False), DEVICE)
save(pd.DataFrame(attn_stats), "attention_stats.csv")

EXTRA_META = {
    "dataset": "CIFAR-10", "class_names": CLASS_NAMES, "patch_size": PATCH_SIZE,
    "n_tokens": int(model.n_patches), "train_config": TRAIN_CONFIG,
    "model_config": MODEL_CONFIG, "calibration_clean_metrics": calib_metrics,
    "n_params": count_params(model), "best_val_score": hist["best_score"],
    "seconds_per_evaluation": per_eval,
}

calibration set (n=9000): {'accuracy': 0.8036666666666666, 'precision': 0.806292239960295, 'recall': 0.8036666666666665, 'f1': 0.8032605918185928, 'roc_auc': 0.9783448696844994}
fixed eval subset (n=1000): {'accuracy': 0.811, 'precision': 0.8126961847405271, 'recall': 0.8109999999999999, 'f1': 0.8102854944263512, 'roc_auc': 0.9774944444444446}
cost: 0.104 s per configuration -> 17.3 min for K=10000
saved results\base_transformer\cifar10\tables\attention_stats.csv (32, 5)


## Fault model

**Definition 1 (attention-head crash).** A head is crashed when its output tensor is
replaced with zeros *after* the attention weights multiply V and *before* head
concatenation and the output projection `W_o`.

Faults are injected **at inference only** - the model is trained clean, then frozen.
After every faulted evaluation all masks are restored to 1.

## STEP 3: single-head crash

Every head is crashed in turn while all others stay alive, so each clean/fault comparison differs in exactly one head.

In [9]:
sh_df = single_head.run_single_head_analysis(
    model=model,
    evaluate=evaluate,
    damage_fn=damage_fn,
    criticality_col=CRIT_COL,
    model_name=MODEL_NAME,
    verbose=True,
)
save(sh_df, "single_head_damage.csv")
sh_df.nlargest(10, "criticality")[["head_name", "criticality", "redundancy", DEG_COL]]

  L0_H0    D_pos_mean = 0.018906


  L0_H1    D_pos_mean = 0.011285


  L0_H2    D_pos_mean = 0.021475


  L0_H3    D_pos_mean = 0.012893


  L0_H4    D_pos_mean = 0.011203


  L0_H5    D_pos_mean = 0.010132


... (24 outputs left out)

  L3_H6    D_pos_mean = 0.023750


  L3_H7    D_pos_mean = 0.018894


saved results\base_transformer\cifar10\tables\single_head_damage.csv (32, 30)


,head_name,criticality,redundancy,D_mean
15,L1_H7,0.066982,1.492950e-11,0.042197
23,L2_H7,0.033044,5.066691e-01,0.017049
19,L2_H3,0.031987,5.224582e-01,0.011662
18,L2_H2,0.030170,5.495873e-01,0.012166
13,L1_H5,0.030041,5.515098e-01,0.011412
25,L3_H1,0.029634,5.575814e-01,0.017060
8,L1_H0,0.029223,5.637251e-01,0.012623
20,L2_H4,0.028844,5.693746e-01,0.009639
10,L1_H2,0.027883,5.837195e-01,0.008550
9,L1_H1,0.026953,5.976153e-01,0.008752


### Layer fragility and vulnerability map

In [10]:
layer_summary = single_head.build_layer_summary(sh_df)
model_summary = single_head.build_model_summary(sh_df, layer_summary, MODEL_NAME, clean_eval)
save(layer_summary, "layer_summary.csv")

plots.vulnerability_map(sh_df, "%s - single-head vulnerability (%s)" % (MODEL_NAME, CRIT_COL),
                        FIGURES / "vulnerability_map.png")
plots.criticality_profile(sh_df, "%s - criticality concentration" % MODEL_NAME,
                          FIGURES / "criticality_profile.png", ylabel=CRIT_COL)

print("most critical head      :", model_summary["most_critical_head"],
      "(criticality %.6f)" % model_summary["max_criticality"])
print("median / max criticality: %.4f" % (model_summary["median_criticality"] / model_summary["max_criticality"]))
print("mean redundancy         : %.4f" % model_summary["mean_redundancy"])
print("fragile layer by mean   :", model_summary["most_fragile_layer_by_mean"])
print("fragile layer by max    :", model_summary["most_fragile_layer_by_max"])
layer_summary

saved results\base_transformer\cifar10\tables\layer_summary.csv (4, 7)


most critical head      : L1_H7 (criticality 0.066982)
median / max criticality: 0.3572
mean redundancy         : 0.6416
fragile layer by mean   : 1
fragile layer by max    : 1


,layer,mean_layer_criticality,max_head_criticality,min_head_criticality,mean_redundancy,n_heads,most_critical_head
0,0,0.014059,0.021475,0.010132,0.790112,8,L0_H2
1,1,0.031402,0.066982,0.019228,0.531195,8,L1_H7
2,2,0.027628,0.033044,0.023318,0.587534,8,L2_H7
3,3,0.022932,0.029634,0.018894,0.657647,8,L3_H1


## STEP 4: criticality, redundancy and severity weights

`C = mean(max(damage,0))`, `R = 1 - C/max(C)`, `W = C/mean(C)`. Critical heads are the top 10% by criticality.

In [11]:
weight_lookup = single_head.build_head_weight_lookup(sh_df)
critical_heads = single_head.critical_head_set(sh_df, C.CRITICAL_TOP_FRACTION)
print("critical heads (top %d%%): %s"
      % (100 * C.CRITICAL_TOP_FRACTION, ["L%d_H%d" % h for h in critical_heads]))
print("heads with near-zero effect:", int((sh_df["criticality"] <= 1e-6).sum()), "of", len(sh_df))
save(sh_df[["head_name", "layer", "head", "criticality", "redundancy", "severity_weight"]],
     "head_weights.csv")
sh_df.nsmallest(5, "criticality")[["head_name", "criticality", "redundancy", DEG_COL]]

critical heads (top 10%): ['L1_H7', 'L2_H7', 'L2_H3']
heads with near-zero effect: 0 of 32
saved results\base_transformer\cifar10\tables\head_weights.csv (32, 6)


,head_name,criticality,redundancy,D_mean
5,L0_H5,0.010132,0.848731,0.002510
4,L0_H4,0.011203,0.832743,0.002899
1,L0_H1,0.011285,0.831519,0.001538
7,L0_H7,0.012301,0.816361,-0.000235
3,L0_H3,0.012893,0.807521,0.001808


## STEP 5: random multi-head fault injection

`N_fail ~ Uniform{1..32}`, heads drawn without replacement and crashed simultaneously, evaluated on the fixed subset.

In [12]:
n_heads = len(get_all_heads(model))
mh_df = multihead.run_random_multihead(
    model=model,
    evaluate=evaluate,
    damage_fn=damage_fn,
    weight_lookup=weight_lookup,
    critical_heads=critical_heads,
    model_name=MODEL_NAME,
    k_trials=C.K_TRIALS,
    seed=C.MULTIHEAD_FAULT_SEED,
    num_layers=model.num_layers,
)
save(mh_df, "random_multihead.csv")
mh_df[["N_fail", "N_crit_fail", "W_fail", "W_fail_mean", DEG_COL]].describe()

    trial 500/10000  8.6 trials/s  eta 18.4 min


    trial 1000/10000  8.6 trials/s  eta 17.4 min


    trial 1500/10000  8.7 trials/s  eta 16.4 min


    trial 2000/10000  8.7 trials/s  eta 15.4 min


    trial 2500/10000  8.7 trials/s  eta 14.4 min


    trial 3000/10000  8.7 trials/s  eta 13.5 min


... (12 outputs left out)

    trial 9500/10000  8.7 trials/s  eta 1.0 min


    trial 10000/10000  8.7 trials/s  eta 0.0 min


saved results\base_transformer\cifar10\tables\random_multihead.csv (10000, 36)


,N_fail,N_crit_fail,W_fail,W_fail_mean,D_mean
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,16.575600,1.561600,16.579578,1.001522,0.302253
std,9.164163,1.100785,9.209998,0.137212,0.200552
min,1.000000,0.000000,0.422094,0.422094,-0.000235
25%,9.000000,1.000000,8.705420,0.946714,0.110383
50%,17.000000,2.000000,16.592587,1.000000,0.293987
75%,24.000000,3.000000,24.501492,1.038926,0.498248
max,32.000000,3.000000,32.000000,2.790351,0.600690


### Degradation by fault fraction and failed-head criticality

In [13]:
grid = multihead.degradation_grid(mh_df, DEG_COL)
save(grid.reset_index(), "degradation_grid.csv")
plots.degradation_grid_plot(
    grid, "%s - mean %s by fault fraction and criticality" % (MODEL_NAME, DEG_LABEL),
    FIGURES / "degradation_grid.png", value_label=DEG_LABEL)
grid

saved results\base_transformer\cifar10\tables\degradation_grid.csv (5, 5)


crit_quartile,Q1,Q2,Q3,Q4
fault_bin,,,,
0-10%,0.012701,0.018044,0.019530,0.028708
10-25%,0.051709,0.065915,0.069236,0.088464
25-50%,0.156714,0.188604,0.214233,0.217182
50-75%,0.359016,0.399982,0.417121,0.417612
75-100%,0.539884,0.560463,0.562465,0.549472


### How well does fault severity explain degradation?

Rank correlations between each fault-severity descriptor and the degradation, plus the `W_fail` / `N_fail` confound.

In [14]:
from scipy.stats import kendalltau, pearsonr, spearmanr

def severity_correlations(df, deg_col):
    rows = []
    for sev in ("N_fail", "L_fault", "N_crit_fail", "W_fail", "W_fail_mean"):
        x, y = df[sev].to_numpy(float), df[deg_col].to_numpy(float)
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        kr, kp = kendalltau(x, y)
        rows.append({"severity": sev, "degradation": deg_col, "pearson": pr,
                     "spearman": sr, "kendall": kr, "spearman_p": sp})
    return pd.DataFrame(rows)

corr_df = severity_correlations(mh_df, DEG_COL)
save(corr_df, "severity_correlations.csv")

# W_fail is partly a restatement of N_fail - report the confound explicitly
conf = spearmanr(mh_df["W_fail"], mh_df["N_fail"]).statistic
print("Spearman(W_fail, N_fail) = %.3f  <- how much of W_fail is just fault count" % conf)
corr_df

saved results\base_transformer\cifar10\tables\severity_correlations.csv (5, 6)
Spearman(W_fail, N_fail) = 0.994  <- how much of W_fail is just fault count


,severity,degradation,pearson,spearman,kendall,spearman_p
0,N_fail,D_mean,0.985756,0.991066,0.927976,0.000000e+00
1,L_fault,D_mean,0.574876,0.651024,0.538535,0.000000e+00
2,N_crit_fail,D_mean,0.805481,0.809946,0.664529,0.000000e+00
3,W_fail,D_mean,0.989203,0.994731,0.937844,0.000000e+00
4,W_fail_mean,D_mean,0.043638,0.145774,0.091595,1.271577e-48


## STEP 6: empirical probabilistic fault-tolerance regions

Widest `(n, c)` with `P(Y <= tau | N_fail <= n, N_crit_fail <= c) >= 0.95`.

In [15]:
grids, safe_summary = saferegion.run_safe_region_analysis(
    mh_df, SAFE_COL, TAUS, n_heads, len(critical_heads),
    min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB,
    tau_labels=TAU_LABELS)
save(grids, "safe_region_grid.csv")
save(safe_summary, "safe_region_summary.csv")

tau_df = saferegion.required_tau_table(
    mh_df, SAFE_COL, n_heads, len(critical_heads),
    min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB)
save(tau_df, "required_tau.csv")
plots.required_tau_plot(
    tau_df, "%s - required degradation threshold for 95%% safety" % MODEL_NAME,
    FIGURES / "required_tau.png", ylabel="tau required (%s)" % SAFE_LABEL)

safe_summary

saved

 results\base_transformer\cifar10\tables\safe_region_grid.csv (384, 9)
saved results\base_transformer\cifar10\tables\safe_region_summary.csv (3, 9)
saved results\base_transformer\cifar10\tables\required_tau.csv (128, 6)


,tau,safe_boundary,n_fail_max,n_crit_fail_max,max_p_safe,coverage_percent,smallest_required_tau,n_boundaries_evaluated,threshold_name
0,0.010,NaN,NaN,NaN,0.000000,0.00,0.030041,128,tau_0.010
1,0.025,NaN,NaN,NaN,0.656489,0.00,0.030041,128,tau_0.025
2,0.050,"N_fail<=2, N_crit_fail<=0",2.0,0.0,0.992263,5.17,0.030041,128,tau_0.050


## STEP 7: summary

In [16]:
summary = {
    "model_name": MODEL_NAME,
    "n_heads": n_heads,
    "n_layers": model.num_layers,
    "k_trials": int(C.K_TRIALS),
    "n_fault_eval_samples": int(C.N_FAULT),
    "clean_metrics": clean_eval,
    "single_head": model_summary,
    "critical_heads": ["L%d_H%d" % h for h in critical_heads],
    "criticality_col": CRIT_COL,
    "degradation_col": DEG_COL,
    "safe_degradation_col": SAFE_COL,
    "severity_correlations": corr_df.to_dict(orient="records"),
    "safe_regions": safe_summary.to_dict(orient="records"),
    "criticality_profile": sorted(sh_df["criticality"].tolist(), reverse=True),
    "median_over_max_criticality": float(sh_df["criticality"].median() / (sh_df["criticality"].max() + 1e-12)),
    "n_heads_zero_effect": int((sh_df["criticality"] <= 1e-6).sum()),
    "worst_single_head": {
        "head": sh_df.loc[sh_df["criticality"].idxmax(), "head_name"],
        "criticality": float(sh_df["criticality"].max()),
        "degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), DEG_COL]),
        "safe_degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), SAFE_COL]),
    },
    **EXTRA_META,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print("wrote", (OUT / "summary.json").relative_to(ROOT))

print()
print("=" * 70)
print("CONCLUSIONS -", MODEL_NAME)
print("=" * 70)
print("clean performance      :", clean_eval)
print("worst single head      : %s  ->  %s = %.6f"
      % (summary["worst_single_head"]["head"], DEG_COL, summary["worst_single_head"]["degradation"]))
print("median/max criticality : %.4f  (1.0 = uniform, 0.0 = fully concentrated)"
      % summary["median_over_max_criticality"])
print("critical heads         :", summary["critical_heads"])
print("layer of critical heads:", sorted({h.split("_")[0] for h in summary["critical_heads"]}))
for r in summary["safe_regions"]:
    print("safe region tau=%-10s: %s (P=%.3f, coverage %.2f%%)"
          % (r.get("threshold_name"), r.get("safe_boundary"), r.get("max_p_safe") or float("nan"),
             r.get("coverage_percent") or 0.0))

wrote results\base_transformer\cifar10\summary.json

CONCLUSIONS - base_transformer_cifar10
clean performance      : {'accuracy': 0.811, 'precision': 0.8126961847405271, 'recall': 0.8109999999999999, 'f1': 0.8102854944263512, 'roc_auc': 0.9774944444444446}
worst single head      : L1_H7  ->  D_mean = 0.042197
median/max criticality : 0.3572  (1.0 = uniform, 0.0 = fully concentrated)
critical heads         : ['L1_H7', 'L2_H7', 'L2_H3']
layer of critical heads: ['L1', 'L2']
safe region tau=tau_0.010 : nan (P=nan, coverage 0.00%)
safe region tau=tau_0.025 : nan (P=0.656, coverage 0.00%)
safe region tau=tau_0.050 : N_fail<=2, N_crit_fail<=0 (P=0.992, coverage 5.17%)
